# small128_strong — strong-corpus capacity stress test

This is a **source-pure historical-master arm**, deliberately separate from the
current vh/flywheel lineage.  It asks a one-sided capacity question: can a
10b×128ch model trained from scratch with the newly promising high-LR recipe
absorb trajectories produced by much stronger players?

Corpus: **13,412,868 states** (3.5× the old `distill_pillar3k` corpus): all V14
pillar3f self-play + crisis, only the genuinely new V15 self-play suffix, all
later 3f crises, and all available pillar3k `crisis_v17` games.  Every state was
relabelled uniformly by `pillar3k_r3_dw3_T0.7_epoch_22` using its exact legal
top-5 FP16 policy.  Original heterogeneous visit targets are not mixed.

Recipe is the current `scratch128_lr3e3` recipe with **only the corpus changed**:
`bs=32768`, `lr=3e-3`, one-epoch warmup, plain cosine, `T=1`, pure soft CE,
unweighted, corrected D4 + color augmentation.  A win establishes capacity; a
failure does not prove incapacity because the historical successful student used
hard/soft blending and different optimizer geometry.

The promoted `small128_vh3` **20K reference bar** is mean **13,765**, P50
**9,658**, P5 **1,140**, P10 **1,908**, `<1000` **3.9%**.  Use short 1K
screens only to choose checkpoints, then compare `small128_strong` and vh3 on
the same 5K seed distribution.  Individual seed trajectories are not paired
evidence in this stochastic game.

Upload to `MyDrive/alphatrain/`:

1. `colorlines_pillar3d_v5.tar.gz` — 1,176,876 bytes
2. `master_3f3k_pillar3k.pt.gz` — 568,591,060 bytes


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, torch
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v5.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v5.tar.gz

# Refuse the historical corrupt D4 implementation.
_dataset_source = open('/content/alphatrain/dataset.py').read()
assert ('def _build_line_direction_luts' in _dataset_source and
        'def _transform_observation' in _dataset_source), (
    'STALE CODE TARBALL: rebuild/upload colorlines_pillar3d_v5.tar.gz')
print('Exact directional-channel D4 transform confirmed.')

os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/master_3f3k_pillar3k.pt.gz /content/master_3f3k_pillar3k.pt.gz
gz=os.path.getsize('/content/master_3f3k_pillar3k.pt.gz')
print(f'.gz: {gz:,} bytes')
assert gz == 568_591_060, f'.gz truncated! got {gz}; re-upload'
!gunzip -t /content/master_3f3k_pillar3k.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/master_3f3k_pillar3k.pt.gz > /content/alphatrain/data/master_3f3k_pillar3k.pt
pt=os.path.getsize('/content/alphatrain/data/master_3f3k_pillar3k.pt')
assert pt == 2_293_607_903, f'.pt size wrong! got {pt}'
d=torch.load('/content/alphatrain/data/master_3f3k_pillar3k.pt',
             map_location='cpu', weights_only=False, mmap=True)
assert d['boards'].shape[0] == 13_412_868
assert d.get('relabeled_by','').endswith('pillar3k_r3_dw3_T0.7_epoch_22.pt')
assert d.get('value_mode') == 'policy_slim'
assert d.get('metadata', {}).get('policy_status') == 'teacher_topk'
assert d.get('metadata', {}).get('policy_precision') == 'fp16'
assert d['pol_indices'].shape == (13_412_868, 5)
print(f'corpus: {pt/1e9:.2f} GB, {len(d["boards"]):,} uniformly relabelled states '
      f'({time.time()-t0:.0f}s)')
del d
!rm /content/master_3f3k_pillar3k.pt.gz
!pip install -q numpy numba scipy


In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG (small128_strong) =====
CHANNELS = 128
EPOCHS   = 40
BATCH    = 32768
LR       = 3e-3
T        = 1.0
DW       = 0
BLEND    = 0.0
GAMMA    = 0  # primary capacity arm is unweighted
SEED     = 42
SAVE_STEPS = 1000
RUN      = "small128_strong"
print(f'RUN={RUN} ep={EPOCHS} bs={BATCH} lr={LR} gamma={GAMMA}')


In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/master_3f3k_pillar3k.pt \
    --channels {CHANNELS} --seed {SEED} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DW} --blend-alpha {BLEND} --disagree-gamma {GAMMA} \
    --save-every-steps {SAVE_STEPS} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/drive/MyDrive/alphatrain/{RUN}_ckpts 2>&1 | tee /content/{RUN}_train.log
# val is informational only — the gate is GAMEPLAY (floor-first, 5k decides).


In [ ]:
# Checkpoints already save DIRECTLY to Drive ({RUN}_ckpts/) —
# every epoch_N.pt and e{E}_s{S}.pt lands there as it is written.
import glob
for f in sorted(glob.glob(f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/*.pt')):
    print(f)


## Gate protocol (M5, C++ eval — download checkpoints as they save)

```bash
python -m alphatrain.inference_cpp.export_ts --model alphatrain/data/iter5a_e1_s400.pt
mv alphatrain/inference_cpp/data/policy_ts.pt alphatrain/inference_cpp/data/dagger1_e1_s400_ts.pt
cd alphatrain/inference_cpp
./build/eval --model data/dagger1_e1_s400_ts.pt --device mps --seed-start 775000 --seed-end 775500 --batch 500
```
1. **Screen step checkpoints** (s100..s900 + epoch saves) at 500 seeds. The 500-seed screen is a
   **catastrophe filter ONLY** — it rejects blowups; it does NOT rank close calls (vh1 itself read
   P50 −6% at 500 seeds, then WON +4.6% at 5k — HISTORY:3375).
2. Take the 2-3 best-looking survivors (floor-first: <1000%, P5/P10) to the **5k eval**
   (`--seed-end 780000`) against vh1's bar: mean 13,080 / P50 9,323 / P5 1,222 / <1000 3.5%.
3. Expected per the MICRO-GO calibration: +3-8%% median. If it clears → name it `small128_vh2`,
   retrain the survival head on ITS backbone, re-export, then RE-GATE Engine B (own-MCTS judge)
   at the new level — the alternation plan (memory: project_small_model_distill).
4. All step checkpoints regress at 500 (catastrophically) → recheck mix %% and label sanity
   before ANY recipe surgery (HISTORY 178 lesson: measure, don't theorize).